# 01 — Signal research: does the order book predict the next move?

First pass over the recorded data. For each candidate signal we ask two questions:

1. **Is it predictive?** Bucket the signal into deciles and look at the average forward mid-price
   return per bucket. A real signal gives a clean monotonic pattern, on every coin.
2. **Is it big enough to trade?** Compare the predicted move with the cost of trading
   (half the spread + fees). Most signals are real but smaller than costs; those can still be
   used to *skew passive quotes* in a market maker rather than to take liquidity.

Candidate signals: L1 imbalance, microprice deviation, 5-level depth imbalance, trade-flow imbalance,
and Binance → Hyperliquid lead-lag.

**Rules for this notebook**
* Set `HOLDOUT_START` once you have ~2 weeks of data and never move it. Everything after it is
  dropped below and stays unseen until a strategy is final.
* Log every variant you try in `research/LOG.md` — you need that count for the deflated Sharpe ratio.

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT))
from hl_recorder import compact, load
from hl_recorder.stats import report
from research import signals as S

DATA = Path(os.environ.get("HLR_DATA", ROOT / "data"))   # recorder --data directory
COINS = ["BTC", "ETH", "SOL", "HYPE"]
COIN = "ETH"                     # coin for the detailed plots
GRID = "100ms"                   # sampling grid
HORIZONS = ["1s", "5s", "30s"]   # forward-return horizons
HOLDOUT_START = None             # e.g. "2026-12-08" -- set once, never move it
TAKER_FEE_BPS = 4.5              # Hyperliquid perps base tier (0.045%), checked Nov 2026
MAKER_FEE_BPS = 1.5              # base tier maker (0.015%); lower with volume tiers / HYPE staking

plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Convert raw hours to Parquet, including the hour still being recorded.
compact(DATA, include_current=True)
print(report(DATA))

## 1. Data quality

Check before trusting anything: message rates over time (gaps show up as dips to zero), and the
latency between exchange timestamp and local receive time. The latency matters later: a backtest
must assume you react at least this late.

In [ ]:
def load_hl(coin):
    out = {ch: S.split_holdout(load(DATA, ch, coin), HOLDOUT_START)
           for ch in ("bbo", "l2Book", "trades", "bn_book5", "bn_trades")}
    return out

raw = {c: load_hl(c) for c in COINS}
d = raw[COIN]
print({k: len(v) for k, v in d.items()})

fig, ax = plt.subplots()
for ch in ("bbo", "l2Book", "trades", "bn_book5", "bn_trades"):
    df = d[ch]
    if df.empty:
        continue
    per_min = pd.to_datetime(df.recv_ns, utc=True).dt.floor("1min").value_counts().sort_index()
    ax.plot(per_min.index, per_min.values, label=ch)
ax.set_yscale("log"); ax.set_title(f"{COIN}: messages per minute"); ax.legend(ncol=5, fontsize=8)
plt.show()

lat = pd.DataFrame({
    ch: (d[ch].recv_ns / 1e6 - d[ch].time_ms).describe(percentiles=[.5, .9, .99])
    for ch in ("bbo", "trades", "bn_book5", "bn_trades") if not d[ch].empty
})
print("receive time - exchange time (ms):")
lat

## 2. Market snapshot

Spread and price path for the detail coin. The spread sets the cost floor: crossing it to trade
costs half the spread on entry and half on exit, plus fees.

In [ ]:
f = S.bbo_features(d["bbo"])
t = pd.to_datetime(f.recv_ns, utc=True)

fig, ax = plt.subplots(1, 2, gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(t, f.mid, lw=0.6); ax[0].set_title(f"{COIN} mid (Hyperliquid)")
ax[1].hist(f.spread_bps.clip(upper=f.spread_bps.quantile(0.99)), bins=60)
ax[1].set_title("spread (bps)")
plt.tight_layout(); plt.show()

spreads = pd.DataFrame({c: S.bbo_features(raw[c]["bbo"]).spread_bps.describe(percentiles=[.5, .9])
                        for c in COINS if not raw[c]["bbo"].empty}).T
spreads[["50%", "90%", "mean"]].rename(columns=lambda c: f"spread {c} (bps)")

## 3. Features on a regular grid

Everything is sampled every `GRID` with backward as-of joins on receive time, so a grid point only
sees data that had arrived by then — no lookahead. Stale values (older than 2 s, e.g. during an
outage) become NaN instead of being carried forward.

In [ ]:
def build_panel(coin, grid_freq=GRID, clock="recv"):
    r = raw[coin]
    if clock == "exchange":
        r = {k: S.exchange_clock(v) for k, v in r.items()}
    if r["bbo"].empty:
        return pd.DataFrame()
    grid = S.make_grid(r["bbo"].recv_ns.min(), r["bbo"].recv_ns.max(), grid_freq)
    p = S.asof(grid, S.bbo_features(r["bbo"]), ["mid", "spread_bps", "imb1", "micro_dev_bps"])
    if not r["l2Book"].empty:
        p = p.join(S.asof(grid, S.book_imbalance(r["l2Book"], levels=5), ["imb5"]))
    for w in ("1s", "5s", "30s"):
        p[f"tfi_{w}"] = S.trade_flow(r["trades"], grid, window=w)
    if not r["bn_book5"].empty:
        bn = r["bn_book5"].assign(bid_px=r["bn_book5"].bid_px.str[0], ask_px=r["bn_book5"].ask_px.str[0])
        bn_mid = S.asof(grid, bn.assign(bn_mid=(bn.bid_px + bn.ask_px) / 2), ["bn_mid"]).bn_mid
        p["bn_mid"] = bn_mid
        # "catch-up" signal: Binance moved over the last second but Hyperliquid did not (yet)
        p["bn_minus_hl_1s"] = S.past_returns(bn_mid, "1s") - S.past_returns(p.mid, "1s")
    p = p.join(S.forward_returns(p.mid, HORIZONS))
    return p

panels = {c: build_panel(c) for c in COINS}
P = panels[COIN]
P.describe().T[["count", "mean", "std", "min", "max"]]

## 4. Is it predictive? Decile plots

Mean forward return (bps) per signal decile, **demeaned**: the sample's average return is
subtracted first, so a trending period doesn't tilt every curve. What to look for: a monotonic
staircase, the same shape at every horizon, and the same shape on every coin (section 5).
Error bars are naive standard errors; grid samples overlap, so they are too optimistic
(see `S.effective_n`).

In [ ]:
FEATURES = ["imb1", "micro_dev_bps", "imb5", "tfi_1s", "tfi_5s", "tfi_30s", "bn_minus_hl_1s"]
FEATURES = [f for f in FEATURES if f in P and P[f].notna().sum() > 1000]

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(13, 3.6), sharey=False)
for ax, h in zip(axes, HORIZONS):
    for feat in FEATURES:
        tab = S.decile_table(P[feat], P[f"fwd_{h}"], demean=True)
        if tab.empty:
            continue
        se = tab.fwd_std_bps / np.sqrt(tab.n)
        ax.errorbar(tab.index, tab.fwd_mean_bps, yerr=se, marker="o", ms=3, lw=1, capsize=2, label=feat)
    ax.axhline(0, color="k", lw=0.5); ax.set_title(f"{COIN}: forward {h}")
    ax.set_xlabel("signal decile")
axes[0].set_ylabel("excess fwd return (bps)")
axes[-1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

## 5. Does it hold across coins? Information coefficients

Spearman rank correlation between each signal and the forward return, per coin. A signal that only
works on one coin is probably noise. The `eff. n` row is a rough count of independent samples at
each horizon — the honest denominator for how much evidence there is.

In [ ]:
rows = {}
for c, p in panels.items():
    if p.empty:
        continue
    for h in HORIZONS:
        rows[(c, h)] = {f: S.ic(p[f], p[f"fwd_{h}"]) for f in FEATURES if f in p}
ic_table = pd.DataFrame(rows).T
ic_table.index.names = ["coin", "horizon"]
eff_n = pd.Series({h: S.effective_n(len(P), h, GRID) for h in HORIZONS}, name="eff. n per coin")
display(ic_table.style.background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3).format("{:.3f}"))
eff_n

## 6. Is it big enough to trade?

For each signal: the average forward move in the top and bottom decile, compared with the cost of
a round trip taking liquidity (full spread + 2 × taker fee) and a one-sided passive entry. If the
edge is smaller than the taker cost but meaningful relative to the spread, the signal is a
candidate for *skewing quotes* in a market maker rather than for taking.

In [ ]:
h = "5s"
med_spread = P.spread_bps.median()
taker_rt = med_spread + 2 * TAKER_FEE_BPS
rows = {}
for feat in FEATURES:
    tab = S.decile_table(P[feat], P[f"fwd_{h}"], demean=True)
    if tab.empty:
        continue
    rows[feat] = {"bottom decile (bps)": tab.fwd_mean_bps.iloc[0],
                  "top decile (bps)": tab.fwd_mean_bps.iloc[-1],
                  "top-bottom spread (bps)": tab.fwd_mean_bps.iloc[-1] - tab.fwd_mean_bps.iloc[0]}
edge = pd.DataFrame(rows).T
print(f"{COIN}, horizon {h}: median spread {med_spread:.2f} bps | taker round trip ≈ {taker_rt:.2f} bps "
      f"| maker fee {MAKER_FEE_BPS:.2f} bps per side")
edge["beats taker round trip?"] = edge["top-bottom spread (bps)"] / 2 > taker_rt
edge

## 7. Lead-lag: does Binance move first?

Correlation between Hyperliquid's 100 ms mid returns and Binance's, shifted by a lag. A peak at a
positive lag means Binance leads.

**Two clocks, two questions** (section 1 shows why: the Hyperliquid feed arrives much later
than Binance's):
* **Receive clock** — what this machine sees. The peak mixes a real price lag with the fact that
  Hyperliquid's feed is slower. It *overstates* the tradeable lead: Hyperliquid's actual book has
  already moved by the time its update reaches you.
* **Exchange clock** — each venue's own timestamps. Closer to the true price lead-lag (assumes
  both exchanges' clocks are in sync to a few ms).

The tradeable window is roughly: exchange-clock lead minus your order latency to Hyperliquid.

In [ ]:
panels_ex = {c: build_panel(c, clock="exchange") for c in COINS}

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
peaks = {}
for ax, (clock, pans) in zip(axes, [("receive", panels), ("exchange", panels_ex)]):
    for c, p in pans.items():
        if p.empty or "bn_mid" not in p or p.bn_mid.notna().sum() < 1000:
            continue
        cc = S.lead_lag(p.mid.ffill(), p.bn_mid.ffill(), max_lag="2s", ret="100ms")
        ax.plot(cc.index, cc.values, marker=".", label=c)
        peaks[(c, clock)] = {"peak lag (s)": cc.idxmax(), "peak corr": cc.max(), "corr at 0": cc.loc[0.0]}
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"{clock} clock")
    ax.set_xlabel("lag (s); > 0 means Binance leads")
axes[0].set_ylabel("corr of 100ms returns"); axes[1].legend()
plt.tight_layout(); plt.show()
pd.DataFrame(peaks).T.unstack().swaplevel(axis=1).sort_index(axis=1)

## 8. Notes and next steps

* **Overlap:** neighbouring grid points share most of their forward window; treat decile error bars
  and t-stats as optimistic. Use `S.effective_n` for honest sample sizes.
* **Regimes:** re-run per day. A signal whose sign flips between days is not a signal.
* **Multiple testing:** every feature × horizon × parameter you try counts. Write them into
  `research/LOG.md`.
* **Clocks:** any lead-lag result has to survive on the exchange clock *and* after adding a
  realistic order latency before it counts as an edge.
* **Next:** pick the most robust signal (consistent across coins and days), then build the
  event-driven backtester for a quote-skewing market maker with a conservative fill model.